# Prepare the Prompt and the Yearly Statement Blocks

| Item | Description |
|---|---|
| Goal | Fix everything the policy-summary run (T3.2.3) needs before any model call. That is the prompt wording, one block of statements per hearing year for the map step, the reduce step that combines the yearly summaries, the token budget check, and the generation settings. |
| Input | `topic68_statements.parquet` from T3.2.1 in the T3.2 Processed folder. |
| Output | `policy_summary_input.json` in the T3.2 Interim folder, read by T3.2.3: the prompts, the settings, and the yearly blocks. `table_map_calls.csv` under `R - Results/tables/...`: one row per map call with its year, statements, and prompt tokens. |
| Model | `Qwen/Qwen3.5-4B`, 262,144-token context. Generation as in T1: `do_sample=True`, `temperature=0.1`, reasoning traces off, at most 512 new tokens per call. |
| Length rule | The final summary must embed whole in bge-large-en-v1.5, so it may have at most 512 encoder tokens. The reduce prompt asks for 200 to 250 words, about 390 tokens; T3.2.3 re-asks shorter if needed. Each yearly summary is asked to be at most 150 words. |
| Run | Run 2026-10-31. The blocks and prompts equal the October 2026 input of the production run; only the model is now named by its Hub id instead of a local folder. |


## Step 1. Import packages


In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer
from IPython.display import Markdown, display


## Step 2. Define file locations and settings

Every number the model run depends on is named here and copied into the input file, so the run never retypes it.


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.2 - Topic Side - Policy Summary of the Legislator Statements")
NOTEBOOK_NAME = "T3.2.2 - Prepare - Prompt and Yearly Statement Blocks"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
STATEMENTS_FILE = PROCESSED_FOLDER / "topic68_statements.parquet"

MODEL = "Qwen/Qwen3.5-4B"
TOPIC_ID = 68
TOPIC_LABEL = "consumer data privacy legislation"
INPUT_TOKEN_BUDGET = 32_000  # model tokens per call; far below the 262,144 context, keeps every call fast on one GPU
MAP_TARGET_WORDS = "at most 150"  # per-year summary; a year with one or two statements should come out much shorter
REDUCE_TARGET_WORDS = "200 to 250"  # lands inside the 512-token limit with headroom (about 1.55 tokens per word)
MAX_NEW_TOKENS = 512  # hard cap on generated tokens per call
MAX_BGE_TOKENS = 512  # the encoder's input limit the final summary must respect
GENERATION = {"do_sample": True, "temperature": 0.1}  # as in T1
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure" / ".env"
for folder in (INTERIM_FOLDER, TABLE_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Point the Hugging Face cache at the scratch root


In [3]:
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)
# A token is needed only for gated models. Qwen3.5-4B and bge-large-en-v1.5 are open; the lines stay so the cell also serves a gated model.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")


## Step 4. Load the statements

Hearing order, so each year's block reads in the order the hearings were held.


In [4]:
statements = pd.read_parquet(STATEMENTS_FILE)
assert statements["text"].notna().all() and statements["stance"].isin(["support", "concern"]).all()
statements = statements.sort_values(["hearing_year", "packageId", "turn_idx", "record_idx"]).reset_index(drop=True)
year_min, year_max = int(statements["hearing_year"].min()), int(statements["hearing_year"].max())
print(f"Statements: {len(statements):,}; hearings {year_min}-{year_max}; by stance: {statements['stance'].value_counts().to_dict()}; years with statements: {statements['hearing_year'].nunique()}")


Statements: 374; hearings 1999-2025; by stance: {'support': 221, 'concern': 153}; years with statements: 26


## Step 5. Format the statement lines

One line per statement: its number, the hearing year, the stance, and the text. The stance is given so the model does not infer it.


In [5]:
statement_lines = [f"[{number}] ({row.hearing_year}, {row.stance}) {row.text}" for number, row in enumerate(statements.itertuples(index=False), start=1)]
statements["number"] = range(1, len(statements) + 1)
statements["line"] = statement_lines
print("\n".join(statement_lines[:3]))


[1] (1999, support) The bill is an important first step that recognizes that as the Internet becomes more of a presence in global commerce there have to be guarantees and assurances that business and personal information remains confidential.
[2] (1999, support) With all due respect, this process is too long, which is why in the E-PRIVACY bill we give the administration a one-time 15-day review of products that are generally available before they are permitted to export them.
[3] (1999, support) The second area where I think the PROTECT Act can go farther is the explicit delineation of the rights and procedural protections of Americans in their ability to use encryption and to be secure in their use of encrypted data.


## Step 6. Define the prompts

The system prompt fixes the role. The map prompt summarizes one year's statements and is told to write less when the year says little. The reduce prompt combines the yearly summaries into the policy summary. It names the two failure modes: a summary that only says the topic is about laws, and knowledge brought in from outside the statements.


In [6]:
SYSTEM_PROMPT = (
    "You are a policy analyst with a working knowledge of United States congressional hearings. "
    "You report only what the statements say."
)

MAP_PROMPT_TEMPLATE = """Below are {n_statements} statements that members of the House Science and Senate Commerce committees made in hearings held in {year} on the topic "{topic_label}". Each statement is numbered and marked with the legislator's stance toward the topic, support or concern.

--- STATEMENTS ---
{statements_block}
--- END OF STATEMENTS ---

Summarize, in {target_words} words, what the legislation discussed in these statements asks of companies and other organizations and what it tries to regulate: which data practices, which kinds of data, and which actors. Write continuous prose. If the statements say little, write only a sentence or two. Do not list or count the statements, do not name legislators, and do not add knowledge beyond the statements."""

REDUCE_PROMPT_TEMPLATE = """Below are {n_years} summaries, one per year, of statements that members of the House Science and Senate Commerce committees made in hearings held between {year_min} and {year_max} on the topic "{topic_label}". Each summary is marked with its year.

--- YEARLY SUMMARIES ---
{summaries_block}
--- END OF YEARLY SUMMARIES ---

Combine them into one summary of the legislation, in {target_words} words, for a reader who has not seen the statements. Cover two things:
1. What the legislation asks of companies and other organizations: the obligations it imposes on them and the rights it gives consumers.
2. What it tries to regulate: which data practices, which kinds of data, and which actors.

Write continuous prose in one or two paragraphs. Name the specific requirements, data practices, and actors; do not say that the topic is about laws, bills, or privacy in general. Do not add knowledge beyond the yearly summaries."""

display(Markdown("**System prompt**\n\n" + SYSTEM_PROMPT + "\n\n**Map prompt template, one call per year**\n\n```\n" + MAP_PROMPT_TEMPLATE
                 + "\n```\n\n**Reduce prompt template, one call**\n\n```\n" + REDUCE_PROMPT_TEMPLATE + "\n```"))


**System prompt**

You are a policy analyst with a working knowledge of United States congressional hearings. You report only what the statements say.

**Map prompt template, one call per year**

```
Below are {n_statements} statements that members of the House Science and Senate Commerce committees made in hearings held in {year} on the topic "{topic_label}". Each statement is numbered and marked with the legislator's stance toward the topic, support or concern.

--- STATEMENTS ---
{statements_block}
--- END OF STATEMENTS ---

Summarize, in {target_words} words, what the legislation discussed in these statements asks of companies and other organizations and what it tries to regulate: which data practices, which kinds of data, and which actors. Write continuous prose. If the statements say little, write only a sentence or two. Do not list or count the statements, do not name legislators, and do not add knowledge beyond the statements.
```

**Reduce prompt template, one call**

```
Below are {n_years} summaries, one per year, of statements that members of the House Science and Senate Commerce committees made in hearings held between {year_min} and {year_max} on the topic "{topic_label}". Each summary is marked with its year.

--- YEARLY SUMMARIES ---
{summaries_block}
--- END OF YEARLY SUMMARIES ---

Combine them into one summary of the legislation, in {target_words} words, for a reader who has not seen the statements. Cover two things:
1. What the legislation asks of companies and other organizations: the obligations it imposes on them and the rights it gives consumers.
2. What it tries to regulate: which data practices, which kinds of data, and which actors.

Write continuous prose in one or two paragraphs. Name the specific requirements, data practices, and actors; do not say that the topic is about laws, bills, or privacy in general. Do not add knowledge beyond the yearly summaries.
```

## Step 7. Build one block per hearing year and check the budget

Each year's map prompt is rendered exactly as the run will render it, with the chat template and reasoning traces off, and counted with the model's tokenizer. Every year must fit the per-call budget.


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
chunks = []
for hearing_year, year_statements in statements.groupby("hearing_year", sort=True):
    user_prompt = MAP_PROMPT_TEMPLATE.format(n_statements=len(year_statements), year=hearing_year, topic_label=TOPIC_LABEL,
                                             statements_block="\n".join(year_statements["line"]), target_words=MAP_TARGET_WORDS)
    rendered = tokenizer.apply_chat_template([{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}],
                                             tokenize=False, add_generation_prompt=True, enable_thinking=False)
    chunks.append({"chunk_id": len(chunks) + 1, "hearing_year": int(hearing_year), "n_statements": int(len(year_statements)),
                   "statement_numbers": year_statements["number"].tolist(), "n_model_tokens": len(tokenizer(rendered)["input_ids"])})
chunk_table = pd.DataFrame(chunks).drop(columns="statement_numbers")
assert chunk_table["n_statements"].sum() == len(statements) and chunk_table["n_model_tokens"].le(INPUT_TOKEN_BUDGET).all()
chunk_table.to_csv(TABLE_FOLDER / "table_map_calls.csv", index=False)
print(f"Map calls: {len(chunks)}, one per year with statements; largest prompt {chunk_table['n_model_tokens'].max():,} model tokens against a budget of {INPUT_TOKEN_BUDGET:,}")
display(chunk_table.set_index("chunk_id").T)


Map calls: 26, one per year with statements; largest prompt 3,512 model tokens against a budget of 32,000


chunk_id,1,2,3,4,5,6,7,8,9,10,...,17,18,19,20,21,22,23,24,25,26
hearing_year,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
n_statements,9,33,15,13,8,4,2,8,1,6,...,21,11,44,40,6,62,1,10,9,6
n_model_tokens,683,1666,1100,1350,648,547,346,606,281,612,...,1496,911,2463,2392,503,3512,235,806,884,508


## Step 8. Save the input file

The JSON file carries the prompts, the settings, and the blocks; T3.2.3 reads it and retypes nothing.


In [8]:
settings = {"model": MODEL, "generation": GENERATION, "enable_thinking": False, "max_new_tokens": MAX_NEW_TOKENS, "map_target_words": MAP_TARGET_WORDS,
            "reduce_target_words": REDUCE_TARGET_WORDS, "max_bge_tokens": MAX_BGE_TOKENS, "input_token_budget": INPUT_TOKEN_BUDGET}
policy_summary_input = {"topic_id": TOPIC_ID, "topic_label": TOPIC_LABEL, "n_statements": len(statements), "year_min": year_min, "year_max": year_max,
                        "mode": "map_by_year_then_reduce", "chunks": chunks, "statement_lines": statement_lines,
                        "prompts": {"system": SYSTEM_PROMPT, "map_template": MAP_PROMPT_TEMPLATE, "reduce_template": REDUCE_PROMPT_TEMPLATE}, "settings": settings}
(INTERIM_FOLDER / "policy_summary_input.json").write_text(json.dumps(policy_summary_input, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved policy_summary_input.json: {len(statement_lines)} statement lines, {len(chunks)} map blocks")


Saved policy_summary_input.json: 374 statement lines, 26 map blocks
